# Lab 05 extra: How prompt caching changes what a long prompt costs

**Going further, for Lab 05**

## Goal

Notebook 2 ended with a fat prompt, every rule of the shop in the system prompt, that sent about 2,400 input
tokens on every call. The API has an answer to a long prompt that never changes: **prompt caching**. Send the
same beginning twice, and the second time most of it is billed at a tenth of the price. Here we watch that happen
in `usage`, price it, find the two ways to lose the discount, and work out what the fat prompt costs over a
conversation of five questions.

## What you will learn

- How to read `cached_tokens` and `cache_write_tokens` in a call's `usage`.
- How to price a call's input from those numbers and the three input prices in `models.env`.
- Why only a long prompt is cached, and why the cached part has to be the start of the request.
- How to work out what a conversation costs with the cache and without it.

## Setup

The same first cell as every lab: it reads the model name and the prices from `models.env` and takes the key from
`.env` without printing it.

In [1]:
from pathlib import Path
import os
from uuid import uuid4
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI(timeout=60, max_retries=2)

## A long prompt that is the same on every call

The Chinook music shop has 20 policy files in `data/corpus/`. A support agent that carries all of them in its
system prompt can answer any question about the rules in one call, and sends all 20 on every call. We build that
prompt. (Notebook 2's fat prompt also held the skill texts and was sent with three lookup tools; this one is
the 20 policies alone, so it is shorter.)

In [3]:
policies = [path.read_text().strip() for path in sorted((ROOT / "data" / "corpus").glob("policy_*.md"))]

FAT = "\n\n".join([
    "You answer the support reps of the Chinook music shop from the policies below. Answer in two or three sentences.",
    *policies,
])

print(len(policies), "policies,", len(FAT), "characters")

20 policies, 6763 characters


## The first call writes the prompt to the cache

When a request starts with the same tokens as an earlier request, the API can reuse the work it already did on
them. The shared beginning is called the **prefix**, and the store of reusable prefixes is the **cache**.

We want to watch the cache fill from empty, and this prompt may have been sent from this account before. So every
call here passes a `prompt_cache_key`. The API keeps a separate cache for each key, and a new key starts empty.
`uuid4().hex[:8]` makes eight random characters, so each run of this notebook gets its own key.

`call` sends one request and returns its `usage`, the part of the response that counts tokens. `messages` is the
conversation after the system prompt.

In [4]:
CACHE_KEY = "desk-" + uuid4().hex[:8]


def call(system, messages, key=CACHE_KEY):
    # One request under a cache key; returns the reply's text and the usage
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}] + messages,
        reasoning_effort=EFFORT or omit,
        prompt_cache_key=key,
    )
    return response.choices[0].message.content, response.usage


QUESTION = [{"role": "user", "content": "A customer was charged twice for the same album on the same day. What do we do?"}]

answer, first = call(FAT, QUESTION)
print(answer)
print()
print("prompt_tokens:", first.prompt_tokens)
print(first.prompt_tokens_details)

If there are two invoices for the same album on the same day, refund the later invoice in full to the original payment method. Ask the customer to provide both invoice numbers and attach them to the refund request; approved refunds take 5–10 business days.

prompt_tokens: 1456
PromptTokensDetails(audio_tokens=0, cache_write_tokens=1453, cached_tokens=0, image_tokens=None, text_tokens=None)


The answer comes from the refunds policy. The numbers under it show what the cache did:

- `prompt_tokens`: 1,456 input tokens, the 20 policies and the question.
- `cache_write_tokens`: 1,453. The API stored the prefix, all but the last 3 tokens of the request.
- `cached_tokens`: 0. Nothing was read from the cache, because this key's cache was empty.

## The same call again reads it back

In [5]:
answer, second = call(FAT, QUESTION)

print("prompt_tokens:", second.prompt_tokens)
print(second.prompt_tokens_details)

prompt_tokens: 1456
PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=1453, image_tokens=None, text_tokens=None)


The same 1,456 input tokens, and the two cache numbers have changed places: `cached_tokens` is 1,453 and
`cache_write_tokens` is 0. The second call found the prefix the first one stored.

**Important:** `prompt_tokens` did not change. The model was sent every token both times. What changed is the price
of 1,453 of them.

## Price a call from its usage

`models.env` has three prices for input tokens, in US dollars per million tokens:

In [6]:
PRICE_INPUT = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_CACHED = float(settings["PRICE_CACHED_INPUT_PER_MILLION"])
PRICE_WRITE = float(settings["PRICE_CACHE_WRITE_PER_MILLION"])

print("ordinary input:", PRICE_INPUT)
print("read from the cache:", PRICE_CACHED)
print("written to the cache:", PRICE_WRITE)

ordinary input: 0.1
read from the cache: 0.01
written to the cache: 0.125


A token read from the cache costs a tenth of an ordinary input token. A token written to the cache costs a quarter
more than an ordinary one: storing is charged for.

Every input token of a call is in exactly one of three groups, so the cost of the input is a sum of three terms:

- **written:** `cache_write_tokens`, at the write price;
- **cached:** `cached_tokens`, at the cached price;
- **ordinary:** what is left, `prompt_tokens - written - cached`, at the input price.

`or 0` covers a model that reports `None` for a field it does not have.

In [7]:
def input_cost(usage):
    # Ordinary input, cache reads and cache writes, each at its own price, in US dollars
    written = usage.prompt_tokens_details.cache_write_tokens or 0
    cached = usage.prompt_tokens_details.cached_tokens or 0
    ordinary = usage.prompt_tokens - written - cached
    return (ordinary * PRICE_INPUT + cached * PRICE_CACHED + written * PRICE_WRITE) / 1_000_000


print(f"with no cache at all: ${first.prompt_tokens * PRICE_INPUT / 1_000_000:.7f}")
print(f"first call (write):   ${input_cost(first):.7f}")
print(f"second call (read):   ${input_cost(second):.7f}")

with no cache at all: $0.0001456
first call (write):   $0.0001819
second call (read):   $0.0000148


In words, with our numbers:

| Call | How its 1,456 input tokens are billed | Input cost |
|---|---|---|
| With no cache | 1,456 ordinary | \$0.0001456 |
| First call | 3 ordinary, 1,453 written | \$0.0001819 |
| Second call | 3 ordinary, 1,453 cached | \$0.0000148 |

The first call costs a quarter more than it would with no cache. The second costs about a tenth. Two calls
together come to about 1.35 times one uncached call instead of 2, and every further call adds a tenth.

## Only the start of the request has to match

The cache works on a prefix. So a different question after the same system prompt should still read the system
prompt from the cache.

In [8]:
OTHER = [{"role": "user", "content": "What hours is the support desk staffed?"}]

answer, third = call(FAT, OTHER)
print(answer)
print(third.prompt_tokens, third.prompt_tokens_details)
print(f"${input_cost(third):.7f}")

Support is staffed Monday through Friday, 09:00–17:00 in the shop’s local time, and is closed on public holidays.
1445 PromptTokensDetails(audio_tokens=0, cache_write_tokens=12, cached_tokens=1430, image_tokens=None, text_tokens=None)
$0.0000161


The answer is about support hours this time. Of the 1,445 input tokens, 1,430 were read from the cache and 12 were written: the system prompt matched what the cache holds, and only the new question, with the formatting around it, was new. The input cost \$0.0000161, close to the repeat of the first question.

Now the opposite: the same policies and the first question, with one short sentence put **in front** of
everything.

In [9]:
answer, moved = call("Today is Monday. " + FAT, QUESTION)

print(moved.prompt_tokens, moved.prompt_tokens_details)
print(f"${input_cost(moved):.7f}")

1460 PromptTokensDetails(audio_tokens=0, cache_write_tokens=1457, cached_tokens=0, image_tokens=None, text_tokens=None)
$0.0001824


1,460 input tokens, none of them read from the cache, and 1,457 written: \$0.0001824, the price of a first call. The three words *Today is Monday.* changed the first tokens of the request, so nothing after them matched the stored prefix, although the 20 policies are the same text as before.

**Key lesson:** put what never changes at the start of a request (instructions, rules, tool schemas) and what
changes at the end (the question, today's date, the customer's name). A date at the top of a system prompt makes
every call a first call.

## A short prompt is never cached

The API caches a prefix only from 1,024 tokens up. A short system prompt, sent twice:

In [10]:
SHORT = "You answer the support reps of the Chinook music shop. Answer in two or three sentences."

for attempt in (1, 2):
    answer, usage = call(SHORT, QUESTION)
    print(usage.prompt_tokens, usage.prompt_tokens_details, f"${input_cost(usage):.7f}")

48 PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None) $0.0000048


48 PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None) $0.0000048


48 input tokens, nothing written and nothing read, both times. Under 1,024 tokens the API does not cache, so each call pays the ordinary price, \$0.0000048. That is still a third of the long prompt's cached call (\$0.0000148): the discount makes a long prompt cheaper than it was, not cheaper than a short one.

## What the long prompt costs over a conversation

One call is not how a desk works. A rep asks a question, then another, and each call sends the system prompt and
the whole conversation so far. We ask five questions in one conversation, under a new cache key so that it starts
from an empty cache, and keep each call's `usage`.

In [11]:
QUESTIONS = [
    "A customer was charged twice for the same album on the same day. What do we do?",
    "What hours is the support desk staffed?",
    "A customer bought a digital album yesterday and changed her mind. Can she get her money back?",
    "A vinyl record arrived cracked. What do we send?",
    "Can a customer change the email on her account twice in one year?",
]

conversation_key = "desk-" + uuid4().hex[:8]
conversation = []
usages = []

for question in QUESTIONS:
    conversation.append({"role": "user", "content": question})
    answer, usage = call(FAT, conversation, key=conversation_key)
    conversation.append({"role": "assistant", "content": answer})
    usages.append(usage)

    details = usage.prompt_tokens_details
    print(f"{usage.prompt_tokens:5} input tokens | written {details.cache_write_tokens or 0:5} | "
          f"cached {details.cached_tokens or 0:5} | ${input_cost(usage):.7f}")

 1456 input tokens | written  1453 | cached     0 | $0.0001819


 1513 input tokens | written    57 | cached  1453 | $0.0000220


 1583 input tokens | written    70 | cached  1510 | $0.0000241


 1642 input tokens | written    59 | cached  1580 | $0.0000235


 1696 input tokens | written    54 | cached  1639 | $0.0000234


Read the five rows from the top:

1. **The first question** wrote the 20 policies to the cache: 1,453 tokens written, \$0.0001819.
2. **Every later question** read from the cache almost everything the call before it had sent (all but its last 3 tokens), and wrote only what was new, the last answer and the new question: 57, 70, 59 and 54 tokens.
3. **The input grows,** from 1,456 to 1,696 tokens, because each call carries the conversation so far. The cost per call stays between \$0.0000220 and \$0.0000241, about an eighth of the first call.

The totals, with the cache and at the ordinary input price:

In [12]:
sent = sum(usage.prompt_tokens for usage in usages)
with_cache = sum(input_cost(usage) for usage in usages)
without_cache = sent * PRICE_INPUT / 1_000_000

print("input tokens sent over five calls:", sent)
print(f"input cost at the ordinary price:  ${without_cache:.7f}")
print(f"input cost with the cache:         ${with_cache:.7f}")
print(f"the cache saved {100 * (1 - with_cache / without_cache):.0f}%")

input tokens sent over five calls: 7890
input cost at the ordinary price:  $0.0007890
input cost with the cache:         $0.0002749
the cache saved 65%


Five calls sent 7,890 input tokens. At the ordinary price that is \$0.0007890. With the cache it came to \$0.0002749, and most of that is the first call's write. The cache saved 65% of the input cost of this conversation, and the share grows with every further question.

**What caching does not change:**

- **The size of the request.** All 7,890 tokens were sent and read by the model. A long prompt still fills the
  model's context, and a model still has to find the one rule it needs among the others.
- **The first call.** It costs more than with no cache. A prompt that is sent once, or that changes near its start
  every time, pays the write price and never gets the discount.
- **Time.** A stored prefix is kept for at least 30 minutes after its last use (the default in the prompt caching
  guide for this model; each read starts the 30 minutes again), sometimes longer (the guide's data retention section gives 24 hours as the maximum).
  After a quiet hour the next call may have to write it again, at the write price.

So skills and caching solve different problems. Skills, in the two notebooks of this lab, make the request smaller.
Caching makes a request that stays the same cheaper. An agent can use both: its instructions, skill list and tool
schemas sit at the start of every call, and once that start passes 1,024 tokens it is cached.

**About the saved outputs.** Every count above is read from the `usage` of a real call to the model `models.env`
names: eleven calls in all. Token counts repeat exactly for the same text; the answers' wording does not. Each run
makes new cache keys, so your run shows the same pattern of writes and reads.

**Checkpoint.** The first call under a new key has `cache_write_tokens` close to `prompt_tokens` and
`cached_tokens` 0, and the second has them the other way round. A different question still reads the system prompt
from the cache; a sentence put in front of the prompt reads nothing. The short prompt shows 0 and 0. Over five
questions the cache cuts the input cost by more than half.

## What you can do now

- Read `cached_tokens` and `cache_write_tokens` from a call's `usage`, and say which tokens were billed at which
  price.
- Price the input of one call and of a whole conversation from `models.env`.
- Order a prompt so that the part that never changes is at the start.
- Say what caching leaves as it was: the size of the request, and what the model has to read.

## Resources

- [OpenAI: prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching): the 1,024-token minimum,
  cache writes and reads, `prompt_cache_key`, how long a prefix is kept
- [OpenAI: pricing](https://developers.openai.com/api/docs/pricing): the input, cached input and cache write
  prices per model
- [OpenAI: Chat Completions API reference](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create):
  `usage` and `prompt_tokens_details`

Back to [notebook 1](05a_meet_context_and_skills.ipynb) or
[notebook 2](05b_context_and_skills_at_the_desk.ipynb).